[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/Pesquisa-Operacional-III-A/blob/main/11_Simulacao_em_Tabelas.ipynb)

## Pesquisa Operacional III-A

Professor: Diogo Ferreira de Lima Silva (TEP-UFF)

# Simulação em Tabelas


Nas aulas teóricas, construímos **simulações em tabelas** preenchendo, linha a linha:

- um **relógio** (o instante atual da simulação);
- uma **fila de eventos** (os eventos futuros já conhecidos);
- as **variáveis de estado** do sistema.

Este notebook implementa em Python **exatamente a mesma mecânica**, para os dois exemplos vistos em sala um exemplo de **linha de montagem** com três máquinas sequenciais;

O objetivo é que você veja a tabela do quadro **nascer de um código**, e, ao final, calcule as estatísticas da forma conceitualmente correta.

Na próxima aula, passaremos a usar a biblioteca **SimPy**, que automatiza esse mesmo motor de eventos.

In [ ]:
import pandas as pd
import numpy as np

## Exemplo da linha de montagem


Relembrando o problema visto em sala: um produto passa por **três máquinas sequenciais** $M_1 \rightarrow M_2 \rightarrow M_3$.
- $M_1$ **nunca fica ociosa** (há sempre insumo disponível).
- Cada máquina processa **uma unidade por vez**.

**Variáveis de estado**

- $L_2$: número de unidades em $M_2$ **+** sua fila;
- $L_3$: número de unidades em $M_3$ **+** sua fila.

**Eventos:**

- `T1`: $M_1$ termina o processamento
- `T2`: $M_2$ termina o processamento
- `T3`: $M_3$ termina o processamento

### Os tempos de processamento

Em sala, usamos uma tabela de números aleatórios já gerada (função `ALEATÓRIOENTRE` do Excel). Vamos usar **os mesmos valores**, para que o resultado do código seja idêntico ao dos slides.

Cada lista abaixo é consumida **na ordem**: a primeira vez que $M_1$ termina, usamos `tempos_M1[0]`; a segunda vez, `tempos_M1[1]`; e assim por diante.

In [ ]:
# Tempos de processamento (mesmos valores da tabela auxiliar usada em sala)
tempos_M1 = [2, 2, 2, 3, 3, 3, 3, 2, 3, 1]
tempos_M2 = [3, 4, 4, 3, 3, 5, 3, 5, 5, 3]
tempos_M3 = [2, 3, 2, 2, 2, 4, 4, 2, 3, 4]

# Conjuntos de onde esses valores vieram (uniforme discreta):
# M1 em {1,2,3}, M2 em {2,3,4}, M3 em {3,4,5}, todos equiprovaveis.

### O motor de simulação

A função abaixo é o coração da simulação. Ela recebe os tempos de processamento e o critério de parada, e devolve a **tabela de simulação** (a mesma do quadro).

Sobre a **fila de eventos**: ela é uma lista de pares `(tempo, tipo)`. A cada passo, ordenamos por tempo e pegamos o primeiro. Quando dois eventos têm o **mesmo tempo**, precisamos de uma regra de desempate. Em sala, adotamos a prioridade `T1` antes de `T2` antes de `T3`. É só uma convenção, mas precisa ser fixada para a tabela ser reproduzível.

In [ ]:
def simular_linha_montagem(tempos_M1, tempos_M2, tempos_M3, parada_relogio=9):
    """
    Simula a linha de montagem de 3 maquinas em serie.

    Parametros
    ----------
    tempos_M1, tempos_M2, tempos_M3 : listas de tempos de processamento
    parada_relogio : a simulacao para quando o relogio ultrapassaria este valor

    Retorna
    -------
    DataFrame com colunas Tempo, Evento, L2, L3 (a tabela de simulacao)
    """
    # --- estado inicial ---
    relogio = 0
    L2, L3 = 0, 0

    # indices que indicam qual sera o proximo tempo a consumir de cada lista
    i1 = i2 = i3 = 0

    # prioridade de desempate quando dois eventos ocorrem no mesmo tempo
    prioridade = {"T1": 1, "T2": 2, "T3": 3}

    # fila de eventos: lista de pares (tempo, tipo)
    # no inicio, so sabemos que M1 vai terminar a 1a unidade
    fila_eventos = [(tempos_M1[i1], "T1")]
    i1 += 1

    # tabela de simulacao: comeca com o estado no tempo 0
    tabela = [(0, "-", L2, L3)]

    while fila_eventos:
        # 1) proximo evento = menor tempo; desempate pela prioridade do tipo
        fila_eventos.sort(key=lambda ev: (ev[0], prioridade[ev[1]]))
        tempo, tipo = fila_eventos[0]

        # criterio de parada: se o proximo evento passa do limite, encerra
        if tempo > parada_relogio:
            break

        # 2) avanca o relogio e remove o evento da fila
        fila_eventos.pop(0)
        relogio = tempo

        # 3) dispara o evento: atualiza estado e agenda eventos futuros
        if tipo == "T1":
            M2_estava_ociosa = (L2 == 0)
            L2 = L2 + 1
            # M1 sempre tem insumo: agenda o proximo termino de M1
            fila_eventos.append((relogio + tempos_M1[i1], "T1"))
            i1 += 1
            # se M2 estava livre, esta unidade comeca a ser processada agora
            if M2_estava_ociosa:
                fila_eventos.append((relogio + tempos_M2[i2], "T2")); i2 += 1

        elif tipo == "T2":
            havia_fila_em_M2 = (L2 > 1)
            M3_estava_ociosa = (L3 == 0)
            L2 = L2 - 1
            L3 = L3 + 1
            # se ainda ha unidade esperando em M2, ela entra em processamento
            if havia_fila_em_M2:
                fila_eventos.append((relogio + tempos_M2[i2], "T2")); i2 += 1

            # se M3 estava livre, a unidade que chegou comeca a ser processada
            if M3_estava_ociosa:
                fila_eventos.append((relogio + tempos_M3[i3], "T3")); i3 += 1

        elif tipo == "T3":
            havia_fila_em_M3 = (L3 > 1)
            L3 = L3 - 1
            if havia_fila_em_M3:
                fila_eventos.append((relogio + tempos_M3[i3], "T3")); i3 += 1

        # 4) registra o novo estado na tabela
        tabela.append((relogio, tipo, L2, L3))

    return pd.DataFrame(tabela, columns=["Tempo", "Evento", "L2", "L3"])

Vamos rodar e comparar com a tabela construída no quadro:

In [ ]:
tabela_montagem = simular_linha_montagem(tempos_M1, tempos_M2, tempos_M3, parada_relogio=9)
tabela_montagem

## Estatísticas a partir da tabela


### Taxa de ocupação de $M_2$

A máquina $M_2$ está **ocupada** sempre que houver pelo menos uma unidade nela ou em sua fila, isto é, $L_2 \geq 1$.

A taxa de ocupação é a fração do tempo total em que $M_2$ esteve ocupada.

In [ ]:
def taxa_ocupacao_M2(tabela):
    """Fração do tempo simulado em que M2 esteve ocupada (L2 >= 1)."""
    tempo_ocupada = 0
    for k in range(1, len(tabela)):
        intervalo   = tabela["Tempo"].iloc[k] - tabela["Tempo"].iloc[k - 1]
        L2_anterior = tabela["L2"].iloc[k - 1]   # estado durante o intervalo
        if L2_anterior >= 1:
            tempo_ocupada += intervalo
    tempo_total = tabela["Tempo"].iloc[-1] - tabela["Tempo"].iloc[0]
    return tempo_ocupada, tempo_total

ocupada, total = taxa_ocupacao_M2(tabela_montagem)
print(f"Tempo com M2 ocupada : {ocupada}")
print(f"Tempo total simulado : {total}")
print(f"Taxa de ocupacao M2  : {ocupada}/{total} = {ocupada/total:.4f}  ({100*ocupada/total:.1f}%)")

### Tamanho médio da fila de $M_2$

Quando $L_2 \geq 1$, uma unidade está **sendo processada** e as demais ($L_2 - 1$) estão **esperando na fila**.

Logo o número de unidades na fila de $M_2$ é $\max(L_2 - 1,\ 0)$.

O **tamanho médio da fila** ($L_q$) é a **média ponderada pelo tempo** dessa quantidade: cada observação de $L_q$ vale mais ou menos conforme o tempo que o sistema permaneceu naquele estado.

$$L_q = \frac{\displaystyle\sum_k L_q(t_k)\;\cdot\;\Delta t_k}{T}$$

- **Numerador** $\sum_k L_q(t_k) \cdot \Delta t_k$: soma de cada valor de $L_q$ multiplicado pela duração do intervalo em que ele vigorou.
- **Denominador** $T$: tempo total simulado.

Isso difere de uma média simples porque os intervalos entre eventos têm durações diferentes — um estado que persiste por 4 minutos deve "pesar" o dobro de um que dura apenas 2 minutos.

In [ ]:
def fila_media_M2(tabela):
    """Numero medio de unidades na fila de M2 (media ponderada pelo tempo)."""
    soma_ponderada = 0   # numerador: soma de Lq × Δt por intervalo
    for k in range(1, len(tabela)):
        intervalo   = tabela["Tempo"].iloc[k] - tabela["Tempo"].iloc[k - 1]
        L2_anterior = tabela["L2"].iloc[k - 1]
        na_fila     = max(L2_anterior - 1, 0)
        soma_ponderada += na_fila * intervalo   # Lq × Δt
    tempo_total = tabela["Tempo"].iloc[-1] - tabela["Tempo"].iloc[0]
    return soma_ponderada, tempo_total

soma_ponderada, total = fila_media_M2(tabela_montagem)
print(f"Σ(Lq × Δt) — soma ponderada no tempo : {soma_ponderada}")
print(f"Tempo total simulado                  : {total}")
print(f"Tamanho medio da fila de M2  (Lq)     : {soma_ponderada}/{total} = {soma_ponderada/total:.4f} unidades")